
<img src="img/viu_logo.png" width="200">

## 01MIAR - Procesamiento de Datos

![logo](img/python_logo.png)

*Ivan Fuertes / Franklin Alvarez*

In [1]:
import numpy as np
import pandas as pd

# Duarante esta sesión mostraremos números flotantes con una precisión de 2 decimales
pd.set_option('display.precision', 2)

### Uniendo datasets con 'join' y 'merge'
- merge() == join()
 - 'join' utiliza por defecto los índices para unir
- Utilizando el parámetro 'on'
 - Si las columnas difieren, 'left_on' y 'right_on'
 
https://miro.medium.com/v2/resize:fit:1400/1*GigXPhr4Ue2zbrgIIoB8Lw.png

### Combinar varios datasets 
- En base a un elemento en común (índice)
- MovieLens 'UserId'

In [5]:
import zipfile as zp # para descomprimir archivos zip
import urllib.request # para descargar de URL
import os

# # descargar MovieLens dataset
url = 'http://files.grouplens.org/datasets/movielens/ml-1m.zip'
local_zip = os.path.join("res", "ml-1m.zip")
urllib.request.urlretrieve(url, local_zip)
 # descomprimiendo archivo zip
with zp.ZipFile(local_zip, 'r') as zipp:
    print('Extracting all files...')
    zipp.extractall(os.path.join("res")) # destino
    print('Done!')

Extracting all files...
Done!


In [6]:
ruta_users = os.path.join("res", "ml-1m", "users.dat")
ruta_ratings = os.path.join("res", "ml-1m", "ratings.dat")
ruta_movies = os.path.join("res", "ml-1m", "movies.dat")

users_dataset = pd.read_csv(ruta_users, sep='::', index_col=0,
    header=None, names=['UserID','Gender','Age','Occupation','Zip-code'], engine='python', encoding="ISO-8859-1")

ratings_dataset = pd.read_csv(ruta_ratings, sep='::', index_col=0,
    header=None, names=['UserID','MovieID','Rating','Timestamp'], engine='python', encoding="ISO-8859-1")

movies_dataset = pd.read_csv(ruta_movies, sep='::', index_col=0,
    header=None, names=['MovieID','Title','Genre'], engine='python', encoding="ISO-8859-1")

In [7]:
display(users_dataset.head(5))
print(len(users_dataset))

,Gender,Age,Occupation,Zip-code
UserID,,,,
1,F,1,10,48067
2,M,56,16,70072
3,M,25,15,55117
4,M,45,7,02460
5,M,25,20,55455


6040


In [8]:
display(ratings_dataset.sample(5))
print(len(ratings_dataset))

,MovieID,Rating,Timestamp
UserID,,,
3965,2247,3,965858821
1980,2612,4,974686648
314,36,2,976473979
2700,3864,1,973304714
2455,3527,3,974180205


1000209


In [9]:
display(movies_dataset.sample(5))
print(len(movies_dataset))

,Title,Genre
MovieID,,
1750,Star Kid (1997),Adventure|Children's|Fantasy|Sci-Fi
1784,As Good As It Gets (1997),Comedy|Drama
1575,Gabbeh (1996),Drama
389,"Colonel Chabert, Le (1994)",Drama|Romance|War
1222,Full Metal Jacket (1987),Action|Drama|War


3883


In [10]:
# combinando users y ratings, ¿Cómo?
combined_dataset = users_dataset.merge(ratings_dataset, on='UserID', how='inner') # parametro 'on' define la columna pivote
display(combined_dataset.head(5))
print(len(combined_dataset))

,Gender,Age,Occupation,Zip-code,MovieID,Rating,Timestamp
UserID,,,,,,,
1,F,1,10,48067,1193,5,978300760
1,F,1,10,48067,661,3,978302109
1,F,1,10,48067,914,3,978301968
1,F,1,10,48067,3408,4,978300275
1,F,1,10,48067,2355,5,978824291


1000209


In [11]:
# combinando movies y el resto
all_dataset = combined_dataset.merge(movies_dataset, on='MovieID', how='inner')
display(all_dataset.head(5))
print(len(combined_dataset))

,Gender,Age,Occupation,Zip-code,MovieID,Rating,Timestamp,Title,Genre
0,F,1,10,48067,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
1,F,1,10,48067,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical
2,F,1,10,48067,914,3,978301968,My Fair Lady (1964),Musical|Romance
3,F,1,10,48067,3408,4,978300275,Erin Brockovich (2000),Drama
4,F,1,10,48067,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy


1000209


### Concatenate
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.concat.html

## Pivot
- Representar los datos en función a varios parámetros, agregando
```python
pivot_table(<lista de valores>, index=<agregador primario>, columns=<agregador secundario>)
```
- https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html
- https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html

In [12]:
display(all_dataset.pivot_table('Rating', index='Gender', columns='Age'))

# Operación equivalente para cada celda
mask = (all_dataset['Gender'] == 'F') * (all_dataset['Age'] == 1)
F1 = all_dataset[mask].Rating.mean()
print("Valor para F-1:", F1)

Age,1,18,25,35,45,50,56
Gender,,,,,,,
F,3.62,3.45,3.61,3.66,3.66,3.80,3.92
M,3.52,3.53,3.53,3.60,3.63,3.69,3.72


Valor para F-1: 3.616290925569276


C:\Users\Acer Predator\AppData\Local\Temp\ipykernel_7992\4040826601.py:4: UserWarning: evaluating in Python space because the '*' operator is not supported by numexpr for the bool dtype, use '&' instead.
  mask = (all_dataset['Gender'] == 'F') * (all_dataset['Age'] == 1)


In [14]:
# Podemos agregar funciones o lista de funciones con las que operar
display(all_dataset.pivot_table('Rating', index='Gender', columns='Age', aggfunc='count'))
display(all_dataset.pivot_table('Rating', index='Gender', columns='Age', aggfunc=['count', 'max', 'mean']))

Age,1,18,25,35,45,50,56
Gender,,,,,,,
F,8827,45427,91340,49473,24110,18064,9199
M,18384,138109,304216,149530,59523,54426,29581


count                                              max        ...     \
Age        1       18      25      35     45     50     56  1  18 25  ... 45   
Gender                                                                ...      
F        8827   45427   91340   49473  24110  18064   9199   5  5  5  ...  5   
M       18384  138109  304216  149530  59523  54426  29581   5  5  5  ...  5   

              mean                                      
Age    50 56    1     18    25    35    45    50    56  
Gender                                                  
F       5  5  3.62  3.45  3.61  3.66  3.66  3.80  3.92  
M       5  5  3.52  3.53  3.53  3.60  3.63  3.69  3.72  

[2 rows x 21 columns]

## Agrupaciones
- agg -> funciones estadísticas de agregación
- Series.unique() -> valores únicos
- pd.value_counts -> ocurrencias

In [ ]:
valores_unicos = all_dataset[mask].Rating.unique()
print(valores_unicos)

In [ ]:
ocurrencias = all_dataset[mask].Rating.value_counts()
ocurrencias.name = "Número de ocurrencias por cada Rating"
print(ocurrencias)

## Manipulación de strings
```python
split(): separar en bloques en función de un carácter
replace(): reemplazar un carácter por otro
index(): encontrar la posición de un carácter
```

### Ejemplo MovieLens: Separar géneros y año en columnas individuales

In [ ]:
# Ejemplo con MovieLens: Genre
## 1: obtener todos los géneros por separado
## 2: crear un dataset de géneros
## 3: por película, marcar género por separado
## 4: Extraer el año de cada peléicula y colocar en columna individual
## 5: eliminar el año del título
## 6: unir con dataset genre
display(movies_dataset.head(3))

In [ ]:
all_genres = movies_dataset['Genre'].apply(lambda x : x.split('|'))

print(all_genres)
# print([genre for x in all_genres for genre in x])
# genres = pd.unique([genre for movie in all_genres for genre in movie]) # TODO Depecrated

# print(all_genres.sum())

genres = pd.unique(all_genres.sum())
print(genres)

In [ ]:
# crear tabla con columnas por género
zeros = np.zeros( (len(movies_dataset), len(genres)) )
genres_frame = pd.DataFrame(zeros, columns=genres, index=list(range(1, len(movies_dataset) + 1)))
display(genres_frame.head(3))

In [ ]:
columns_genres = genres_frame.columns # lista de generos (columnas)
print(columns_genres)
# para cada película, marcar género con 1
for i, genre in enumerate(movies_dataset['Genre']):
    inds = columns_genres.get_indexer(genre.split('|')) # retorna los indices correspondientes a los generos de cada pelicula
    genres_frame.iloc[i,inds] = 1 # localiza las columnas del genero correspondiente, marca con 1

In [ ]:
display(genres_frame.head(5))

In [ ]:
# unir con dataset original
movies_split_genre = movies_dataset.join(genres_frame)

In [ ]:
display(movies_split_genre.head(5))

#### Replace e index para extraer el año de la película

In [ ]:
movies_dataset = pd.read_csv(ruta_movies, sep='::', index_col=0,
    header=None, names=['MovieID','Title','Genre'], engine='python', encoding="ISO-8859-1")
display(movies_dataset.sample(5))

In [ ]:
display(movies_dataset.head(2))

In [ ]:
# extraer el año de la columna Title
def split_year(title):
    index = title.index('(')
    return title[index:].replace('(','').replace(')','')

# crear nueva columna Year
movies_dataset['Year'] = movies_dataset['Title'].apply(split_year)
display(movies_dataset.sample(2))

In [ ]:
# eliminar el año de la columna Title
def remove_year(title):
    index = title.index('(')
    return title[:index-1].rstrip()

movies_dataset['Title'] = movies_dataset['Title'].apply(remove_year)
display(movies_dataset.head(20))

#### Expresiones regulares

- https://docs.python.org/3/library/re.html
- https://regex101.com/

- import re

In [ ]:
# ¿Cómo localizar que 'Zip-code' tiene un formato erróneo?
users_dataset.sample(5)

In [ ]:
# users_dataset['Zip-code'].str.match('^[0-9]{5}$')

display(users_dataset[users_dataset['Zip-code'].str.match('^\d{5}$') == False])

# ^\d{5}$
# ^ = start of the string
# \d = decimal string
# {5} = 5 repeticiones de decimales
# $ = end of string

In [ ]:
movies_dataset = pd.read_csv(ruta_movies, sep='::', index_col=0,
    header=None, names=['MovieID','Title','Genre'], engine='python', encoding="ISO-8859-1")
display(movies_dataset.head(2))

In [ ]:
# ¿Cómo extraer el año con regex en el formato adecuado?
display(movies_dataset['Title'].str.extract('(\d{4})'))

# (\d{4})
# (= busca apertura parentesis
# \d = decimal string
# {4} = 4 repeticiones de decimales
# ) = cierre de parentesis

## Operaciones con colecciones
```python
reduce: aplicar una operación y retornar un valor
map: aplicar  una operación y retornar una secuencia
filter: retorna una secuencia con elementos que cumplen una condición
```


## Reduce
- Aplicar una operación matemática a cada uno de los elementos de una colección
- Diferente de 'apply()' porque retorna un valor numérico
- Ejemplo: Detección de géneros en años específicos

https://docs.python.org/3/library/functools.html

In [15]:
from functools import reduce # necesario para reduce

lista = [1, 3, 5, 7, 9]
print(reduce(lambda x,y: x + y, lista))

25


In [ ]:
movies_1975 = movies_split_genre[ movies_split_genre['Title'].str.contains('1975') ]
movies_1975.head(3)

In [ ]:
any_drama = reduce(lambda x,y : bool(x) | bool(y), movies_1975['Drama']) # hay algún drama en 1975
print(any_drama)

all_comedy = reduce(lambda x,y : bool(x) & bool(y),movies_1975['Comedy']) # son todas las películas de 1975 comedias?
print(all_comedy)

In [ ]:
print(movies_1975['Drama'].any()) # Comprueba si hay algún valor que puede cumplir
print(movies_1975['Comedy'].all()) # Comprueba si todos los valores son True

In [ ]:
# Observar el tipo de dato antes para ver si es posible aplicar las funciones
print(movies_1975.dtypes)
print(movies_1975['Comedy'].unique())

## Filter
- retorna una secuencia con elementos que cumplen una condición
- Ejemplo: obtener las películas de 1975 que contienen 'The' en el título

In [ ]:
filtro = filter(lambda x : 'The' in x, movies_1975['Title'])
print(list(filtro))
# ¿Están todos los títulos con "The"? si tiene mayúsculas o no...

In [ ]:
filtro = filter(lambda x : 'the' in x, movies_1975['Title'].str.lower())
list(filtro)

## Map
- aplicar  una operación y retornar una secuencia
- Cambiar el valor integral de la columna 'Comedy' por bool

In [ ]:
mapa = map(lambda x : bool(x), movies_split_genre['Comedy'])
movies_split_genre.loc[:,'Comedy'] = list(mapa)   # TODO Future warning
display(movies_split_genre.head(4))

## Transformación de variables (calidad de datos)
- Valores no definidos
- Valores duplicados
- Discretización (valores categóricos)

In [17]:
matrix = pd.DataFrame(np.random.randint(10,size=(5,10)))
matrix[matrix < 2] = np.nan
display(matrix)

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2
2,8,6.0,NaN,8.0,4,NaN,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,NaN,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


In [18]:
# nulos por columna
print(matrix.isnull().sum(axis=0))
# display(matrix.isna().sum(axis=0))

0    0
1    1
2    2
3    1
4    0
5    1
6    0
7    3
8    1
9    0
dtype: int64


In [19]:
# Cantidad valores nulos
print(matrix.isnull().sum(axis = 1).sum())

9


In [20]:
# numero de no nulos por fila
print(matrix.count(axis=1))

0     8
1     7
2     7
3     9
4    10
dtype: int64


In [21]:
# Número de nulos por fila
print(matrix.shape[1] - matrix.count(axis=1))

0    2
1    3
2    3
3    1
4    0
dtype: int64


In [22]:
# Representación de las filas en las que una determinada columna tiene nulos
display(matrix[matrix[3].isnull()])

,0,1,2,3,4,5,6,7,8,9
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2


In [23]:
# Conteo de valores que aparecen en el dataset
valores = [3, 4]
# Identificación de valores de dominio que se encuentran en un listado
display(matrix[matrix[3].isin(valores)])

,0,1,2,3,4,5,6,7,8,9
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


In [24]:
display(matrix)

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2
2,8,6.0,NaN,8.0,4,NaN,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,NaN,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


In [25]:
## Tratamiento de valores nulos
# eliminar
display(matrix.dropna(axis=1))

,0,4,6,9
0,8,4,7,4
1,9,8,5,2
2,8,4,9,8
3,5,7,5,8
4,3,5,8,4


In [26]:
# eliminar si no hay un número de valores no NaN
display(matrix)
display(matrix.dropna(thresh=4, axis= 1))

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2
2,8,6.0,NaN,8.0,4,NaN,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,NaN,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


,0,1,3,4,5,6,8,9
0,8,7.0,6.0,4,3.0,7,4.0,4
1,9,NaN,NaN,8,3.0,5,7.0,2
2,8,6.0,8.0,4,NaN,9,6.0,8
3,5,6.0,2.0,7,6.0,5,NaN,8
4,3,4.0,4.0,5,5.0,8,2.0,4


In [27]:
# sustituir por un valor fijo
display(matrix.fillna(-1))

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,-1.0,6.0,4,3.0,7,-1.0,4.0,4
1,9,-1.0,3.0,-1.0,8,3.0,5,-1.0,7.0,2
2,8,6.0,-1.0,8.0,4,-1.0,9,-1.0,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,-1.0,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


In [29]:
# sustituir por valor dinámico (copia)
display(matrix)

display(matrix.bfill())
display(matrix.ffill())

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2
2,8,6.0,NaN,8.0,4,NaN,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,NaN,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


,0,1,2,3,4,5,6,7,8,9
0,8,7.0,3.0,6.0,4,3.0,7,7.0,4.0,4
1,9,6.0,3.0,8.0,8,3.0,5,7.0,7.0,2
2,8,6.0,4.0,8.0,4,6.0,9,7.0,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,2.0,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,7.0,3.0,6.0,8,3.0,5,NaN,7.0,2
2,8,6.0,3.0,8.0,4,3.0,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,6.0,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


In [30]:
# sustituir por valor dinámico (interpolación)
display(matrix)
display(matrix.interpolate())

,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,NaN,3.0,NaN,8,3.0,5,NaN,7.0,2
2,8,6.0,NaN,8.0,4,NaN,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,NaN,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


,0,1,2,3,4,5,6,7,8,9
0,8,7.0,NaN,6.0,4,3.0,7,NaN,4.0,4
1,9,6.5,3.0,7.0,8,3.0,5,NaN,7.0,2
2,8,6.0,3.5,8.0,4,4.5,9,NaN,6.0,8
3,5,6.0,4.0,2.0,7,6.0,5,7.0,4.0,8
4,3,4.0,2.0,4.0,5,5.0,8,6.0,2.0,4


#### Tratar valores duplicados

In [31]:
serie = pd.Series(['a','b','c','a','c','a','g'])
print(serie.duplicated())

0    False
1    False
2    False
3     True
4     True
5     True
6    False
dtype: bool


In [32]:
df = all_dataset
display(df.head(3))

# Eliminación de los duplicados en una columna definida
df2 = df.drop_duplicates(subset="Gender", keep='first', inplace=False)
display(df2)

,Gender,Age,Occupation,Zip-code,MovieID,Rating,Timestamp,Title,Genre
0,F,1,10,48067,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
1,F,1,10,48067,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical
2,F,1,10,48067,914,3,978301968,My Fair Lady (1964),Musical|Romance


,Gender,Age,Occupation,Zip-code,MovieID,Rating,Timestamp,Title,Genre
0,F,1,10,48067,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
53,M,56,16,70072,1357,5,978298709,Shine (1996),Drama|Romance


#### Discretización (valores categóricos)
- Tras Series y DataFrame, objeto para categorías: Categorical
```python
categorias = pd.cut(<valores>, <bins>) 
```

In [33]:
# especificar los bloques
bins = [0,18,35,65,99, np.inf]
edades = [16,25,18,71,44,100,12]
categorias = pd.cut(edades,bins)
print(categorias)

[(0.0, 18.0], (18.0, 35.0], (0.0, 18.0], (65.0, 99.0], (35.0, 65.0], (99.0, inf], (0.0, 18.0]]
Categories (5, interval[float64, right]): [(0.0, 18.0] < (18.0, 35.0] < (35.0, 65.0] < (65.0, 99.0] < (99.0, inf]]


In [34]:
categorias.value_counts()

(0.0, 18.0]     3
(18.0, 35.0]    1
(35.0, 65.0]    1
(65.0, 99.0]    1
(99.0, inf]     1
Name: count, dtype: int64

In [35]:
# especificar el número de bloques
bins = 3
edades = [0,6,8,16,25,18,71,44,100]
categorias = pd.cut(edades,bins) # rangos idénticos (similar distancia de rangos)
print(categorias)
print(categorias.value_counts())

[(-0.1, 33.333], (-0.1, 33.333], (-0.1, 33.333], (-0.1, 33.333], (-0.1, 33.333], (-0.1, 33.333], (66.667, 100.0], (33.333, 66.667], (66.667, 100.0]]
Categories (3, interval[float64, right]): [(-0.1, 33.333] < (33.333, 66.667] < (66.667, 100.0]]
(-0.1, 33.333]      6
(33.333, 66.667]    1
(66.667, 100.0]     2
Name: count, dtype: int64


In [36]:
bins = 3
edades = [1,6,8,16,25,18,71,44,100]
categorias = pd.qcut(edades,bins) # rangos homogéneos (similar número de valores)
print(categorias)
print(categorias.value_counts())

[(0.999, 13.333], (0.999, 13.333], (0.999, 13.333], (13.333, 31.333], (13.333, 31.333], (13.333, 31.333], (31.333, 100.0], (31.333, 100.0], (31.333, 100.0]]
Categories (3, interval[float64, right]): [(0.999, 13.333] < (13.333, 31.333] < (31.333, 100.0]]
(0.999, 13.333]     3
(13.333, 31.333]    3
(31.333, 100.0]     3
Name: count, dtype: int64
